# ChestAI -- M1 inference entry point

The same shared pipeline core the training notebook uses (`chestct.preprocessing.pipeline.process_scan`), run the way the clinical product would: one scan, or a folder of several, no labels, no manifest, nothing persisted to disk.

`INPUT_PATH` can be:
- a single NIfTI file (`.nii` / `.nii.gz`),
- a single DICOM series folder,
- or a folder holding several scans (several patients/series) -- every one found is processed; one bad or QC-failing scan does not stop the rest.

Each scan gets its own row: `passed=False` (no tensor) on a hard error or a failed QC check, `passed=True` with a ready `(K, 3, H, W)` tensor otherwise. Feeding that tensor to an actual classification model is M4's job, once it exists.

## 1. Upload and extract the project zip

In [4]:
import shutil
from pathlib import Path

zip_candidates = sorted(Path("/content").glob("*.zip"))
if not zip_candidates:
    from google.colab import files
    print("No zip found in /content yet -- pick your project zip to upload:")
    uploaded = files.upload()
    zip_candidates = [Path("/content") / name for name in uploaded.keys()]

zip_path = zip_candidates[0]
print("Using zip:", zip_path)

Using zip: /content/ChestCT-Multi-Abnormality-Classifier.zip


In [5]:
import zipfile

PROJECT_DIR = Path("/content/ChestCT-Multi-Abnormality-Classifier")
extract_dir = Path("/content/_extracted")

for d in (PROJECT_DIR, extract_dir):
    if d.exists():
        shutil.rmtree(d)
extract_dir.mkdir()

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(extract_dir)

entries = list(extract_dir.iterdir())
if len(entries) == 1 and entries[0].is_dir():
    shutil.move(str(entries[0]), str(PROJECT_DIR))
else:
    shutil.move(str(extract_dir), str(PROJECT_DIR))

print("Project extracted to:", PROJECT_DIR)

Project extracted to: /content/ChestCT-Multi-Abnormality-Classifier


## 2. Install

In [6]:
%cd /content/ChestCT-Multi-Abnormality-Classifier
!pip install -q ".[dev]"
!python -c "import torch" 2>/dev/null || pip install -q torch

/content/ChestCT-Multi-Abnormality-Classifier
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.1 MB/s eta 0:00:00


If a DICOM folder later fails to decode (compressed pixel data), set this to `True`, run it, and re-run from step 4.

In [7]:
INSTALL_DICOM_CODECS = False #@param {type:"boolean"}
if INSTALL_DICOM_CODECS:
    !pip install -q pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg

## 3. Check the compute available

In [8]:
import torch
print("CUDA available:", torch.cuda.is_available())
DEVICE = "auto" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

CUDA available: True
device: auto


## 4. Point at a scan (or a folder of scans)

For a Drive path that was *shared with you*: in Google Drive, right-click the folder -> **Organize -> Add shortcut** -> My Drive first (Colab's mount only shows My Drive).

In [10]:
INPUT_PATH = "/content/drive/MyDrive/NHRD - CT Scan" #@param {type:"string"}

from pathlib import Path
if INPUT_PATH.startswith("/content/drive") and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
if not Path(INPUT_PATH).exists():
    raise FileNotFoundError(f"{INPUT_PATH!r} was not found")
print("input:", INPUT_PATH)

Mounted at /content/drive
input: /content/drive/MyDrive/NHRD - CT Scan


## 5. Run inference

In [11]:
try:
    from chestct.preprocessing.preprocess_config import PreprocessConfig
    from chestct.inference import run_inference
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "chestct is not installed in THIS runtime. This is not a bug in the code -- "
        "it happens whenever the Colab runtime restarts or disconnects (idle timeout, "
        "a crash, or a 'Restart session' prompt after a pip install), which wipes "
        "every installed package even though the notebook still shows old cell output. "
        "Fix: re-run cells 1-4 from the top of THIS notebook (upload/extract the zip, "
        "pip install, compute check, pick the input path) in the current runtime, then "
        "run this cell again."
    ) from exc

results = run_inference(INPUT_PATH, cfg=PreprocessConfig(device=DEVICE))
print(f"{len(results)} scan(s) found")
for r in results:
    if r.passed:
        print(f"  OK      {r.volume_id}: tensor {tuple(r.pixel_values.shape)}")
    elif r.error:
        print(f"  ERROR   {r.volume_id}: {r.error}")
    else:
        print(f"  REJECTED {r.volume_id}: failed QC -- {'; '.join(r.qc.reasons)}")

7 scan(s) found
  OK      4203-26_P00001_S0001_s002534.1: tensor (194, 3, 224, 224)
  OK      4213-26_P00001_S0001_s318449.1: tensor (150, 3, 224, 224)
  OK      4214-26_P00001_S0001_s469091.1: tensor (230, 3, 224, 224)
  OK      4215-26_P00001_S0001_s000275.1: tensor (207, 3, 224, 224)
  OK      4216-26_P00001_S0001_s021310.1: tensor (220, 3, 224, 224)
  OK      4217-26_P00001_S0001_s979548.1: tensor (177, 3, 224, 224)
  OK      4222-26_P00001_S0001_s791893.1: tensor (201, 3, 224, 224)


## 6. Results table

In [12]:
import pandas as pd
rows = []
for r in results:
    rows.append({
        "volume_id": r.volume_id,
        "scan_path": r.scan_path,
        "format": r.format,
        "passed": r.passed,
        "reasons": "; ".join(r.qc.reasons) if r.qc else (r.error or ""),
        "flags": "; ".join(r.qc.flags) if r.qc else "",
    })
pd.DataFrame(rows)

,volume_id,scan_path,format,passed,reasons,flags
0,4203-26_P00001_S0001_s002534.1,/content/drive/MyDrive/NHRD - CT Scan/4203-26/...,dicom,True,,
1,4213-26_P00001_S0001_s318449.1,/content/drive/MyDrive/NHRD - CT Scan/4213-26/...,dicom,True,,
2,4214-26_P00001_S0001_s469091.1,/content/drive/MyDrive/NHRD - CT Scan/4214-26/...,dicom,True,,
3,4215-26_P00001_S0001_s000275.1,/content/drive/MyDrive/NHRD - CT Scan/4215-26/...,dicom,True,,
4,4216-26_P00001_S0001_s021310.1,/content/drive/MyDrive/NHRD - CT Scan/4216-26/...,dicom,True,,
5,4217-26_P00001_S0001_s979548.1,/content/drive/MyDrive/NHRD - CT Scan/4217-26/...,dicom,True,,
6,4222-26_P00001_S0001_s791893.1,/content/drive/MyDrive/NHRD - CT Scan/4222-26/...,dicom,True,,
